# TurboQuant-v3 Demo

This notebook demonstrates the TurboQuant-v3 quantization algorithm for LLM quantization.

In [ ]:
# Install turboquant if not already installed
!pip install -e .. --quiet

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from turboquant import (
    QuantConfig,
    TurboQuantConfig,
    turboquant_v3_compress,
    turboquant_v3_decompress,
    compute_metrics,
    QuantizedLinear,
)

print("TurboQuant-v3 loaded successfully!")

## 1. Basic Quantization Example

In [ ]:
# Create a random weight matrix (typical of LLM linear layers)
np.random.seed(42)
torch.manual_seed(42)

W = np.random.normal(0, 0.02, size=(512, 1024)).astype(np.float32)
print(f"Original weight shape: {W.shape}")
print(f"Original size: {W.nbytes / 1024:.2f} KB")

In [ ]:
# Configure quantization
config = QuantConfig(
    group_size=64,          # Quantize in groups of 64 columns
    outlier_keep_ratio=0.02, # Keep 2% of channels in FP16
    rank=8,                 # SVD correction rank
    activation_aware=True    # Use activation-aware scaling
)

# Compress weights
comp = turboquant_v3_compress(W, config)

# Show compressed weights info
print(f"Compressed weight shape: {comp.packed_int4.shape}")
print(f"Number of groups: {len(comp.scales)}")
print(f"Outlier channels: {len(comp.protected_indices) if comp.protected_indices is not None else 0}")

In [ ]:
# Decompress and reconstruct
W_rec = turboquant_v3_decompress(comp)

# Compute metrics
metrics = compute_metrics(W, W_rec)

print("\n=== Reconstruction Metrics ===")
print(f"MSE: {metrics['mse']:.8f}")
print(f"Max Error: {metrics['max_error']:.6f}")
print(f"Relative Error: {metrics['relative_error']:.6f}")
print(f"PSNR: {metrics['psnr_db']:.2f} dB")

## 2. PyTorch QuantizedLinear

In [ ]:
# Create a PyTorch linear layer
linear = nn.Linear(1024, 512, bias=True)

# Quantize it
config = QuantConfig(group_size=64, rank=8)
quantized = QuantizedLinear.from_linear(linear, config=config)

print(f"Original layer: {linear}")
print(f"\nQuantized layer: {quantized}")

In [ ]:
# Use the quantized layer
x = torch.randn(4, 1024)  # Batch of 4, 1024 features

output = quantized(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Output dtype: {output.dtype}")

In [ ]:
# Get compression statistics
stats = quantized.get_weight_stats()

print("\n=== Compression Statistics ===")
print(f"Original size: {stats['original_size_bytes'] / 1024:.2f} KB")
print(f"Quantized size: {stats['quantized_size_bytes'] / 1024:.2f} KB")
print(f"Compression ratio: {stats['compression_ratio']:.2f}x")
print(f"Group size: {stats['group_size']}")
print(f"SVD correction: {stats['has_svd_correction']}")
print(f"Protected channels: {stats['has_protected_channels']}")

## 3. Comparing Different Configurations

In [ ]:
configs = [
    {"name": "Group-32, SVD-8", "config": QuantConfig(group_size=32, rank=8)},
    {"name": "Group-64, SVD-8", "config": QuantConfig(group_size=64, rank=8)},
    {"name": "Group-128, SVD-8", "config": QuantConfig(group_size=128, rank=8)},
    {"name": "Group-64, SVD-0", "config": QuantConfig(group_size=64, rank=0)},
    {"name": "Group-64, SVD-16", "config": QuantConfig(group_size=64, rank=16)},
]

results = []
for cfg in configs:
    comp = turboquant_v3_compress(W, cfg["config"])
    W_rec = turboquant_v3_decompress(comp)
    metrics = compute_metrics(W, W_rec)
    
    original_size = W.nbytes
    quantized_size = comp.packed_int4.nbytes + comp.scales.nbytes
    if comp.zero_points is not None:
        quantized_size += comp.zero_points.nbytes
    if comp.svd_u is not None:
        quantized_size += comp.svd_u.nbytes + comp.svd_v.nbytes
    
    results.append({
        "name": cfg["name"],
        "compression": original_size / quantized_size,
        "mse": metrics["mse"],
        "psnr": metrics["psnr_db"]
    })

print("{:<20} {:>12} {:>12} {:>10}".format(
    "Config", "Compression", "MSE", "PSNR (dB)"
))
print("-" * 56)
for r in results:
    print("{:<20} {:>11.2f}x {:>12.8f} {:>10.2f}".format(
        r["name"], r["compression"], r["mse"], r["psnr"]
    ))

## 4. HuggingFace Integration (Conceptual)

In [ ]:
# Example: HuggingFace model quantization (requires transformers)
# from transformers import AutoModelForCausalLM
# from turboquant import TurboQuantConfig, quantize_model

# # Load model
# model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b-hf")

# # Configure quantization
# quant_config = TurboQuantConfig(
#     bits=4,
#     group_size=128,
#     version="gemm",
#     activation_aware=True,
#     rank=8,
# )

# # Quantize
# quantized_model = quantize_model(model, quantization_config=quant_config)

# print("Model quantized successfully!")

print("HuggingFace integration example (commented out for demo)")

## Summary

TurboQuant-v3 provides:
- **~4x compression** with minimal accuracy loss
- **Group-wise INT4 quantization** for fine-grained precision control
- **AWQ-style activation-aware scaling** for better reconstruction
- **Protected FP16 channels** for outlier-sensitive dimensions
- **Optional SVD correction** to recover quantization error
- **Easy PyTorch integration** via QuantizedLinear
- **HuggingFace compatibility** via TurboQuantizer